# Select an implementation with the backend factory

The factory maps a component class and a dataset backend class to an
implementation. The call site keeps the same interface on pandas and Spark.
Use this pattern when an extension needs backend-specific operations. If
`Dataset` already provides the operation on both backends, calling it
directly is simpler.

- [List and search built-in registrations](#built-in)
- [Register and run a custom component](#custom-component)

<a id="built-in"></a>

## 1. List and search built-in registrations

`repr(backend_factory)` gives the number of registered component classes
and backend mappings. `print_registry()` lists every mapping available in
the current session as **component → backend → implementation**. The list
reflects the components imported so far.

In [1]:
import os

# Local tutorial settings. Set these before importing PySpark or HypEx.
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ.setdefault("PYARROW_IGNORE_TIMEZONE", "1")

from math import isclose

from hypex.dataset import Dataset, TargetRole
from hypex.dataset.backends import PandasDataset, SparkDataset
from hypex.extensions.faiss import FaissExtension
from hypex.utils import BackendsEnum
from hypex.utils.registry import BackendFactory, backend_factory

data = Dataset(data={"data": {"revenue": [2.0, 4.0, 9.0]}}, roles={"revenue": TargetRole(float)})

In [2]:
print(repr(backend_factory))
backend_factory.print_registry()

<BackendFactory bases=12 registrations=23>
Key class – BiasExtension:
	Backend – PandasDataset → PandasBisaExtesion
	Backend – SparkDataset → SparkBisaExtesion
Key class – DummyEncoderExtension:
	Backend – PandasDataset → PandasDummyEncoderExtension
	Backend – SparkDataset → SparkDummyEncoderExtension
Key class – FaissExtension:
	Backend – PandasDataset → PandasFaissExtension
	Backend – SparkDataset → SparkFaissExtension
Key class – GroupKSTestExtension:
	Backend – PandasDataset → PandasKSTestExtension
	Backend – SparkDataset → SparkKSTestExtension
Key class – GroupChi2TestExtension:
	Backend – PandasDataset → PandasChi2TestExtension
	Backend – SparkDataset → SparkChi2TestExtension
Key class – MatchingMetricsExtension:
	Backend – PandasDataset → PandasMatchingMetricsExtension
	Backend – SparkDataset → SparkMatchingMetricsExtension
Key class – LstsqExtension:
	Backend – PandasDataset → PandasLstsqExtension
	Backend – SparkDataset → SparkLstsqExtension
Key class – TTest:
	Backend – Panda

To narrow the list, search the registry snapshot by component or implementation
name. Then use `resolve_backend` with the component class and your
`Dataset` to find the implementation HypEx selects for that input.

In [3]:
query = "faiss"
for component, registrations in backend_factory.registry.items():
    for backend_class, implementation in registrations.items():
        if query.casefold() in component.__name__.casefold() or (
            query.casefold() in implementation.__name__.casefold()
        ):
            print(f"{component.__name__} → {backend_class.__name__} → {implementation.__name__}")

resolved = backend_factory.resolve_backend(FaissExtension, data)
print("Selected for this dataset:", resolved.__name__ if resolved else "unregistered")

FaissExtension → PandasDataset → PandasFaissExtension
FaissExtension → SparkDataset → SparkFaissExtension
Selected for this dataset: PandasFaissExtension


### Restore the component interface for type checking

`resolve_backend()` returns a generic class type because it serves many
components. Once you check that a registration exists, cast the selected
class to `Type[FaissExtension]`, using the original registry key as the
type. The IDE can then offer the constructor and method signatures and
documentation from the shared `FaissExtension` interface. Registered
implementations use that same interface. The cast is a static hint and
leaves the selected class and its runtime docstrings intact.

In [4]:
from typing import Type, cast

if resolved is None:
    raise NotImplementedError(f"FAISS does not support {data.backend_type}")

faiss_class = cast(Type[FaissExtension], resolved)
faiss = faiss_class(n_neighbors=1)
print(f"Runtime implementation: {type(faiss).__name__}")

Runtime implementation: PandasFaissExtension


<a id="custom-component"></a>

## 2. Register and run a custom component

Build a small component that estimates mean revenue. The same `calc` method
resolves the implementation on every call. A private `BackendFactory` keeps
the notebook's registrations independent of the built-in ones. The example
raises a clear error when it encounters an unregistered backend.

The pandas implementation below uses HypEx aggregation. The Spark
implementation shows the boundary for custom native code by using Spark SQL
inside the adapter. It collects one aggregate row.

In [5]:
from pyspark.sql import functions as F

factory = BackendFactory()

class MeanRevenue:
    @classmethod
    def calc(cls, data, column):
        implementation = factory.resolve_backend(cls, data)
        if implementation is None:
            raise NotImplementedError(f"{cls.__name__} does not support {data.backend_type}")
        return implementation.compute(data, column)


@factory.register(MeanRevenue, PandasDataset)
class PandasMeanRevenue:
    @staticmethod
    def compute(data, column):
        return float(data.select(column).mean())


@factory.register(MeanRevenue, SparkDataset)
class SparkMeanRevenue:
    @staticmethod
    def compute(data, column):
        # Native Spark operation, isolated inside the backend implementation.
        return float(data.data.agg(F.avg(F.col(column)).alias("mean")).first()["mean"])

In [6]:
from pathlib import Path
from tempfile import TemporaryDirectory


from pyspark.sql import SparkSession

# Configure the local JVM before Spark starts, while keeping error messages visible.
spark_files = TemporaryDirectory(prefix="hypex-spark-")
log_config = Path(spark_files.name) / "log4j2.properties"
log_config.write_text(
    "rootLogger.level = error\n"
    "rootLogger.appenderRef.console.ref = console\n"
    "appender.console.type = Console\n"
    "appender.console.name = console\n"
    "appender.console.target = SYSTEM_ERR\n"
    "appender.console.layout.type = PatternLayout\n"
    "appender.console.layout.pattern = %p %c: %m%n\n",
    encoding="utf-8",
)
spark = (
    SparkSession.builder.master("local[2]")
    .appName("HypEx BackendFactory")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.driver.extraJavaOptions", f"-Dlog4j.configurationFile={log_config.as_uri()}")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [7]:
pandas_mean = MeanRevenue.calc(data, "revenue")

try:
    distributed = data.to_backend(BackendsEnum.spark, session=spark)
    spark_mean = MeanRevenue.calc(distributed, "revenue")
    # Both implementations should calculate the same mean for the same input.
    assert isclose(pandas_mean, spark_mean)
    print(f"pandas mean: {pandas_mean:.2f}")
    print(f"Spark mean: {spark_mean:.2f}")

    factory.unregister(MeanRevenue, SparkDataset)
    try:
        MeanRevenue.calc(distributed, "revenue")
    except NotImplementedError as error:
        print(error)
    else:
        # Success here would mean dispatch still used the removed registration.
        raise AssertionError("The unregistered Spark implementation should not resolve.")
finally:
    try:
        spark.stop()
    finally:
        spark_files.cleanup()

pandas mean: 5.00
Spark mean: 5.00
MeanRevenue does not support BackendsEnum.spark


In a package, registration needs to happen before the first call, usually
when the module containing the implementation is imported. Use
`register_explicit` for a class that has already been defined elsewhere.
See [Custom pipelines](../04_Custom_Pipelines.ipynb) for the executor and
reporter interfaces that surround a custom calculation.